# Combined EDA — German grid load and residual load

**Spec:** [`.claude/specs/03-combined-cherry-picked-eda.md`](../../.claude/specs/03-combined-cherry-picked-eda.md)
(sections `03.1`–`03.7`) · **Data:** `data/smard.csv` (SMARD / Bundesnetzagentur, hourly, region DE)

## What this notebook is

The team's four individual explorations — [`EDA-hari.ipynb`](EDA-hari.ipynb),
[`EDA-magc.ipynb`](EDA-magc.ipynb), [`EDA-robert.ipynb`](EDA-robert.ipynb) and
[`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb) — consolidated into one notebook that runs
top to bottom against `data/smard.csv` alone. Only plot cells **tagged** in a source notebook are
included, every "duplicate" and "combine with X" tag is resolved into a decision recorded in the
relevant sub-spec, and everything is rebuilt against one shared frame, naming convention and unit
system rather than four different ones. The four source notebooks are not modified.

## Sections

| # | Sub-spec | Section |
|---|---|---|
| 1 | [`03.1`](../../.claude/specs/03.1-setup.md) | Setup — loading, helpers, `YEARS`, `LOADED` |
| 2 | [`03.2`](../../.claude/specs/03.2-sanity-check.md) | Sanity check |
| 3 | [`03.3`](../../.claude/specs/03.3-univariate-and-time-structure.md) | Univariate and time structure |
| 4 | [`03.4`](../../.claude/specs/03.4-two-series-comparison-views.md) | Two-series comparison views |
| 5 | [`03.5`](../../.claude/specs/03.5-calendar-structure-heatmaps.md) | Calendar structure and single events |
| 6 | [`03.6`](../../.claude/specs/03.6-temporal-dependence-and-extremes.md) | Temporal dependence, decomposition, ramps and extremes |
| 7 | [`03.7`](../../.claude/specs/03.7-context-appendix.md) | Context appendix, findings, self-check |

## Conventions

- The shared hourly frame is **`time_series`**, never `ts`. `SERIES` names the eight data
  columns, `DERIVED` the ten calendar/helper columns — the split keeps `.corr()` and
  `.describe()` from treating `year` or `dow` as measurements.
- **No literal calendar year in code.** The record's extent is a snapshot, not a constant (the
  team intends to widen the fetch back to 2019), so year lists, anchor years and colour maps are
  derived from `YEARS` at run time. The one deliberate exception is §5's Euro 2024 fixture dates,
  which are historical fact rather than a property of the record.
- **No thresholds.** This notebook defines no risk flag, cut-off or labelled column. Where it
  shows "the tail hours" it selects them by rank, for description only.
- **One colour configuration.** §1.2 assigns a colour to every feature once, so the same series
  is the same colour in every figure of this notebook. No plot picks its own colours.
- **Descriptive slices stay local.** Rank- or date-based selections are computed inside their own
  plotting cell and never persisted onto `time_series`; §7's self-check proves mechanically that
  no flag column crept in.

---

## 1 — Setup

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](../API-connection.ipynb) top to bottom — it pulls the
SMARD API (no key required) and writes the file in German Excel CSV format
(`sep=";"`, `decimal=","`, `utf-8-sig`).

The data directory is resolved by walking **upward** from the working directory rather than by a
fixed `"../../data/smard.csv"`: this notebook sits two levels below the repo root, and a
hardcoded depth breaks as soon as the kernel starts somewhere else (nbconvert from the root, for
instance).

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0  # this notebook draws ~30 figures on purpose

# Walk up from the working directory to the first parent holding a `data/` folder, so the
# notebook runs unmodified from notebooks/01_eda/ (Jupyter) or from the repo root (nbconvert).
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")

### 1.1 — Helpers

Copied unchanged in behaviour from [`EDA-simple-claude.ipynb`](EDA-simple-claude.ipynb), which
holds the project's canonical copies (originally from [`EDA-robert.ipynb`](EDA-robert.ipynb),
corrected there). They are defined **once, here**, and reused — not redefined — by every later
section.

- `_complete_periods` holds the edge rule in exactly one place, shared by both aggregation
  helpers. The rule is *drop periods the data does not fully cover*, which is not the same as
  "drop the first and the last": this record starts on a month boundary, so its opening month is
  complete and only the trailing one goes. A plain `.resample()` produces fake edge dips.
- `style_timeseries` and `seasonal_plot` both require `ylabel`, and `seasonal_plot` actually
  applies it (the original hard-coded `"MWh"` and discarded the argument).

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last".
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW). Deliberately not
        ``mwh_per_day / 24``: a month containing the spring DST switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so a comparison table needs no second copy of this
        arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state whether it shows MWh, average MW or MWh/day.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over tens of thousands of rows — minutes of runtime,
            meaningless band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

### 1.2 — Colour configuration

**The one place colours are decided.** Every figure in this notebook reads its colours from the
mappings below, so a series looks the same wherever it appears and no plot invents its own
palette.

The base palette is [`EDA-hari.ipynb`](EDA-hari.ipynb)'s, with two team decisions applied on top:
**`grid_load` is red** and **`residual_load` near-black** — the two headline series, kept maximally
distinct from each other and from the generation series. Each SMARD day-ahead forecast takes its
measured counterpart's colour and is drawn **dashed**, so a forecast and its actual read as the
same quantity rather than as two unrelated lines.

Year-coded plots call `year_colors(...)`, which samples a continuous colormap over whatever years
the record happens to contain — no fixed year-to-colour table, which would break the moment the
fetch range widens.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Base palette, from EDA-hari.ipynb.
COLORS = {
    "navy":   "#17365D",
    "blue":   "#2C6EBA",
    "teal":   "#4AA3A5",
    "green":  "#2F8F5B",
    "mint":   "#6FBF9B",
    "gold":   "#D9A53A",
    "orange": "#E95D0F",
    "coral":  "#E76F51",
    "red":    "#B10F0F",
    "purple": "#9274B8",
    "gray":   "#AEB8C5",
    "black":  "#1C1C1C",
    "ink":    "#17365D",
    "muted":  "#707B8C",
    "grid":   "#E3E8EF",
    "cream":  "#F7F3E7",
}

# One colour per feature. Forecasts deliberately share their measured counterpart's colour.
SERIES_COLOR = {
    "grid_load":         COLORS["red"],
    "residual_load":     COLORS["black"],
    "wind_on":           COLORS["teal"],
    "wind_off":          COLORS["blue"],
    "solar":             COLORS["gold"],
    "renewables":        COLORS["green"],
    "wind_total":        COLORS["purple"],
    "fc_grid_load":      COLORS["red"],
    "fc_res":            COLORS["black"],
    "fc_gen_wind_solar": COLORS["green"],
}

SERIES_LABEL = {
    "grid_load":         "Grid load",
    "residual_load":     "Residual load",
    "wind_on":           "Onshore wind",
    "wind_off":          "Offshore wind",
    "solar":             "Solar",
    "renewables":        "Wind + solar",
    "wind_total":        "Wind (on + offshore)",
    "fc_grid_load":      "Forecast grid load",
    "fc_res":            "Forecast residual load",
    "fc_gen_wind_solar": "Forecast wind + solar",
}

FORECAST_COLS = ["fc_gen_wind_solar", "fc_grid_load", "fc_res"]
HEADLINE_COLS = ["grid_load", "residual_load"]  # drawn slightly heavier than the rest


def series_style(col, **overrides):
    """Line kwargs for `col`: colour, width and dash pattern, from the configuration above."""
    style = {
        "color": SERIES_COLOR[col],
        "linewidth": 2.4 if col in HEADLINE_COLS or col in FORECAST_COLS[1:] else 1.8,
        "linestyle": "--" if col in FORECAST_COLS else "-",
        "label": SERIES_LABEL[col],
    }
    style.update(overrides)
    return style


def year_colors(years, cmap="viridis"):
    """A colour per year, sampled from `cmap` — never a fixed year-to-colour table."""
    years = list(years)
    sampled = plt.get_cmap(cmap)(np.linspace(0.05, 0.95, len(years)))
    return dict(zip(years, sampled))


# Heatmap scales. `grid_load` is strictly positive and takes a sequential ramp towards its own
# red; `residual_load` crosses zero and needs a diverging, zero-centred scale — they cannot share
# one colour bar.
GRID_LOAD_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_grid_load",
    ["#FBF1EF", "#F0C3B8", "#DE8B76", COLORS["coral"], COLORS["red"]],
)
RESIDUAL_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_residual",
    [COLORS["green"], "#B9DCC7", "#FFFFFF", "#8A8A8A", COLORS["black"]],
)
# Hari's remaining scales, reused unchanged by the plots ported from that notebook.
SUPPORT_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_support",
    ["#F7F9FC", "#D7E6F3", "#82B6D9", COLORS["blue"], COLORS["navy"]],
)
DIV_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_diverging",
    [COLORS["teal"], "#DCEEEF", "#FFFFFF", "#F8D1C5", COLORS["coral"]],
)
WIND_SOLAR_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_wind_solar",
    ["#F8F3E7", "#F3C879", "#9BD0C3", COLORS["teal"], COLORS["green"]],
)

print(f"{len(SERIES_COLOR)} features coloured, {len(COLORS)} base colours")

### 1.3 — Load and prepare

The CSV is German Excel format, so every numeric column arrives as text with a comma decimal
separator. The failure mode to guard against is silent: unconverted columns land as a string
dtype, every aggregate still computes something, and every number is wrong. The dtype assertion
below is the guard — a positive `is_float_dtype` test rather than `!= object`, because under
pandas 3 an unconverted column lands as `StringDtype`, which `!= object` would wave straight
through.

The flat, `RangeIndex`ed frame is called `raw` and is **deleted** at the end of the loading
cells. Everything downstream uses `time_series`, so the two cannot drift apart.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them. Note the inconsistent
# capitalisation in the source ("Grid Load" vs "Forecast Grid load") — reproduced deliberately.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid load": "fc_grid_load",
    "Forecast Residual Load": "fc_res",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cells

time_series.head(3)

In [ ]:
print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
display(time_series.dtypes.to_frame("dtype"))

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot for §7's closing self-check, taken before any other cell can touch the frame: a cell
# inserted anywhere in between that mutates `time_series` is caught regardless of which vintage
# of the CSV was loaded.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}
print(f"\nLOADED = {LOADED}")

display(time_series.describe().T)

### 1.4 — Derived columns, `SERIES` / `DERIVED`, and `YEARS`

All derived columns are defined **here, in one place**, immediately after loading. Two of them
are needed by the sanity check itself (`hour` for the night-solar test, `renewables` as the
wind+solar aggregate), so they cannot wait for the section that first plots them.

`YEARS` is computed from the loaded data and is the only permitted source of year information in
the rest of the notebook — no section may write a calendar year into code (see the note on the
one deliberate exception at the top).

`spans_gap` marks the row *following* a gap in the hourly index. The record's only gaps are the
spring DST switches, where the local hour 02:00 does not exist; `.diff()`, `.shift()` and rolling
windows all need that flag, so it is created here with everything else.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_res",
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")
time_series[DERIVED].head(3)

### 1.5 — Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. §7 re-runs the same invariants at the end of the notebook, plus a
comparison against `LOADED`, which together prove that nothing in between mutated the frame.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")